# Week 13 Assignment: Auditing and Cleaning the Semester Dataset

This week you'll apply the data cleaning workflow independently to the semester transactions dataset.
This is the same dataset you'll use for the project in Weeks 14–16, so the quality of your cleaning work here matters.

**You will:**
1. Load and audit the transactions and merchant tables
2. Identify data quality problems across three categories
3. Build a merchant-to-category lookup dictionary
4. Fix each problem using Python
5. Verify your work and save a cleaned file

**Submission:** Push your completed notebook and `transactions_clean.csv` to your GitHub repo inside a `week13/` folder. Paste your repo link in Blackboard.

---
**Dataset files needed** (place these in `week13/data/`):
- `transactions.csv`
- `merchants.csv`
- `categories.csv`
- `accounts.csv`

---
## Part 1: Load the Data

Load all four CSV files into DataFrames and print the shape of each.
Use paths from the `data` folder: `data/transactions.csv`, etc.

In [96]:
import pandas as pd
import numpy as np

# Load all four tables and print their shapes
accounts = pd.read_csv('data/accounts.csv')
categories = pd.read_csv('data/categories.csv')
merchants = pd.read_csv('data/merchants.csv')
transactions = pd.read_csv('data/transactions.csv')

# Print the shapes of each table
print("Accounts shape:", accounts.shape)
print("Categories shape:", categories.shape)
print("Merchants shape:", merchants.shape)
print("Transactions shape:", transactions.shape)

Accounts shape: (5, 3)
Categories shape: (20, 3)
Merchants shape: (27, 5)
Transactions shape: (1378, 9)


In [97]:
# Preview the first 10 rows of the transactions table
# Your code here
print(transactions.head(10))


  transaction_id        date account_id transaction_type category_id  \
0       T0000001  2022-01-01        CHK           Income         PAY   
1       T0000002  2022-01-01        CHK         Transfer      RET401   
2       T0000003  2022-01-01     INV401         Transfer      RET401   
3       T0000617  2022-01-01        CHK          Expense        GROC   
4       T0000473  2022-01-03        CHK          Expense        PHON   
5       T0000618  2022-01-03        CC1          Expense         GAS   
6       T0000377  2022-01-03        CHK          Expense        RENT   
7       T0000569  2022-01-04        CC1          Expense        SUBS   
8       T0000425  2022-01-05        CHK          Expense        UTIL   
9       T0000521  2022-01-05        CHK          Expense         INS   

  merchant_id   amount  payment_method        description  
0   M_PAYROLL  4259.26  Direct Deposit           Paycheck  
1      M_401K  -255.56         Payroll  401k Contribution  
2      M_401K   255.56     

In [98]:
# Preview the merchants table
# Your code here
print(merchants.head(10))


  merchant_id    merchant_name            industry           city  is_online
0   M_PAYROLL          Payroll             Banking         Online          1
1      M_SIDE      Side Hustle             Banking         Online          1
2   M_BANKINT    Bank Interest             Banking         Online          1
3    M_CC_PAY     CC Processor             Banking         Online          1
4      M_401K  401(k) Provider             Banking         Online          1
5       M0001        FreshMart           Groceries     Neenah, WI          0
6       M0002        FreshMart           Groceries  Green Bay, WI          0
7       M0003        QuickStop           Groceries    Oshkosh, WI          0
8       M0004        QuickStop           Groceries   Kimberly, WI          0
9       M0005        BrightGas  Gas/Transportation   Appleton, WI          0


---
## Part 2: Audit

Profile the dataset before making any changes. Your goal is to understand the shape of the data
and flag anything that looks wrong.

Run all three audit checks below, then document what you found in the markdown cell at the end of this section.
**You are looking for problems — there are some to find.**

In [99]:
# Audit check 1: Missing values
# Count missing values in each column.
# Note: empty strings in category_id won't show up as NaN — check for those separately.
# Hint: (transactions['category_id'] == '').sum()
# Your code here
print("Missing values by column: category_id", (transactions['category_id'] == '').sum())
print("Missing values by column:", transactions.isnull().sum())

Missing values by column: category_id 0
Missing values by column: transaction_id      0
date                0
account_id          0
transaction_type    0
category_id         3
merchant_id         0
amount              2
payment_method      0
description         0
dtype: int64


In [100]:
# Audit check 2: Amount ranges
# Use describe() to get a summary of the amount column.
# Then break it down by category to see typical ranges for each type of transaction.
# Hint: groupby('category_id')['amount'].agg(['min', 'median', 'max', 'count'])
#
# Look carefully at the output. For each expense category, ask yourself:
# does the minimum or maximum value make sense given what that category represents?
# Your code here
print('Amount ranges:')
print(transactions['amount'].describe())
print(transactions.groupby('category_id')['amount'].agg(['min', 'median', 'max', 'count']))

Amount ranges:
count     1376.000000
mean       310.820661
std       1366.264916
min      -1671.550000
25%       -106.907500
50%        -56.405000
75%         67.390000
max      10756.080000
Name: amount, dtype: float64
                 min    median       max  count
category_id                                    
BONUS        6303.41  7447.460  10756.08      4
CC_PAY      -1671.55     0.000   1671.55    192
DINE         -210.00   -43.930    -15.00    126
ENT          -220.00   -40.030    -10.00     84
GAS          -106.43   -56.840     -0.41    148
GROC         -842.50   -71.020    -20.00    208
INS          -195.00  -180.000   -165.00     48
INT            21.32    39.215     56.82     16
PAY          4259.26  4615.380   5000.00    105
PHON          -95.00   -90.000    -85.00     48
RENT        -1350.00 -1272.500  -1200.00     48
RET401       -300.00     0.000    300.00    210
SIDE          313.35   487.690    759.73     41
SUBS          -26.00   -24.000    -20.00     47
UTIL        

In [101]:
# Audit check 3: Merchant ID validity
# Compare the merchant IDs used in transactions against the merchants reference table.
# Hint: build a set of valid IDs from merchants['merchant_id'],
# then find IDs in transactions that are not in that set.
# Your code here
inventory_ids = set(transactions['merchant_id'])
valid_ids = set(merchants['merchant_id'])

print('Merchant IDs in transactions:')
print(sorted(inventory_ids))

print('\nValid merchant IDs:')
print(sorted(valid_ids))

# Compute explicit orphaned ID values so audit findings are unambiguous
orphaned_ids = sorted(inventory_ids - valid_ids)
print('\nOrphaned merchant IDs found:')
print(orphaned_ids)

Merchant IDs in transactions:
['M0001', 'M0002', 'M0003', 'M0004', 'M0005', 'M0006', 'M0007', 'M0008', 'M0009', 'M0010', 'M0011', 'M0012', 'M0013', 'M0014', 'M0015', 'M0021', 'M0022', 'M0075', 'M0099', 'M0150', 'M_401K', 'M_BANKINT', 'M_CC_PAY', 'M_PAYROLL', 'M_SIDE']

Valid merchant IDs:
['M0001', 'M0002', 'M0003', 'M0004', 'M0005', 'M0006', 'M0007', 'M0008', 'M0009', 'M0010', 'M0011', 'M0012', 'M0013', 'M0014', 'M0015', 'M0016', 'M0017', 'M0018', 'M0019', 'M0020', 'M0021', 'M0022', 'M_401K', 'M_BANKINT', 'M_CC_PAY', 'M_PAYROLL', 'M_SIDE']

Orphaned merchant IDs found:
['M0075', 'M0099', 'M0150']


**Document your audit findings here.**

Be specific — list each problem type, how many rows are affected, and what made you flag it.

- Missing or empty values:
- Suspicious amounts (list each category and why you flagged it):
- Merchant ID problems:

*3 category ID's were missing, and 2 amounts had empty values.
Dine, Ent, Groc, all have a huge jump in their min compared to the median and max.
There are three merchant ID's in the transactions that are orphaned. 


---
## Part 3: Identify

For each problem you found in Part 2, write code to isolate the specific rows.
Print enough columns to confirm you've found the right rows.

You decide how many cells to use — one per problem type makes sense.
For amount problems: you decide what counts as implausible based on what you saw in the audit.
Document your reasoning for any thresholds you choose.

In [102]:
# Identify rows with missing or empty category_id
# Hint: combine isnull() and == '' with the | operator
# Your code here
print(transactions[transactions['category_id'].isnull() | (transactions['category_id'] == '')])

   transaction_id        date account_id transaction_type category_id  \
12       T0000975  2022-01-08        CC2          Expense         NaN   
80       T0000634  2022-03-26        CHK          Expense         NaN   
83       T0000572  2022-04-02        CC1          Expense         NaN   

   merchant_id  amount payment_method description  
12       M0009  -59.27           Card      Dining  
80       M0004  -83.56          Debit   Groceries  
83       M0011  -20.00            ACH        SUBS  


In [103]:
# Identify rows with missing amount
# Your code here
print(transactions[transactions['amount'].isnull()])

    transaction_id        date account_id transaction_type category_id  \
35        T0000978  2022-02-05        CC2          Expense        DINE   
180       T0000659  2022-07-04        CC1          Expense         GAS   

    merchant_id  amount payment_method description  
35        M0009     NaN           Card      Dining  
180       M0008     NaN           Card         Gas  


In [115]:
# Identify rows with implausible amounts
#
# Based on your audit findings, decide which categories have suspicious values
# and what thresholds make sense. Add a comment explaining your reasoning.
#
# Example reasoning (do not use this — write your own based on what you found):
# "Category X typically ranges from $A to $B based on the median and typical
#  spending patterns. Values outside this range are implausible."
#
# Your code here Dine, Ent, Groc
print('DINE')
print(transactions.loc[transactions['category_id'] == 'DINE', 'amount'].head(10))
print('ENT')
print(transactions.loc[transactions['category_id'] == 'ENT', 'amount'].head(10))
print('GROC')
print(transactions.loc[transactions['category_id'] == 'GROC', 'amount'].head(10))
print("DINE ranges from $15.00 to $91.00 based on the median and spending patterns. This means that values outside this range are implausible .")
print("ENT ranges from $10.00 to $70.00 based on the median and spending patterns. This means that values outside this range are implausible .")
print("GROC ranges from $20.00 to $122.00 based on the median and spending patterns. This means that values outside this range are implausible .")


DINE
26    -36.83
35       NaN
45    -54.52
49    -77.92
64    -15.00
71    -82.88
86    -26.73
93    -46.46
105   -59.32
114   -55.56
Name: amount, dtype: float64
ENT
19     -62.68
46     -43.37
70     -59.64
81     -36.70
87     -67.19
99    -187.50
107    -57.14
115    -10.00
126    -40.75
129    -63.62
Name: amount, dtype: float64
GROC
3     -62.29
11    -89.15
16    -90.94
18    -85.85
25    -32.74
36    -56.47
39    -66.46
44   -105.42
48   -124.50
63    -29.94
Name: amount, dtype: float64
DINE ranges from $15.00 to $91.00 based on the median and spending patterns. This means that values outside this range are implausible .
ENT ranges from $10.00 to $70.00 based on the median and spending patterns. This means that values outside this range are implausible .
GROC ranges from $20.00 to $122.00 based on the median and spending patterns. This means that values outside this range are implausible .


In [105]:
# Identify rows with orphaned merchant IDs
# Hint: use ~isin() against your set of valid merchant IDs
# Your code here
print(transactions[~transactions['merchant_id'].isin(valid_ids)])

    transaction_id        date account_id transaction_type category_id  \
70        T0000983  2022-03-13        CC2          Expense         ENT   
86        T0000986  2022-04-02        CC2          Expense        DINE   
171       T0001003  2022-07-02        CC2          Expense        DINE   

    merchant_id  amount payment_method    description  
70        M0099  -59.64           Card  Entertainment  
86        M0150  -26.73           Card         Dining  
171       M0075  -22.25           Card         Dining  


---
## Part 4: Build the Merchant-to-Category Lookup Dictionary

Before fixing the problems, build a dictionary that maps each `merchant_id` to its correct `category_id`.
You'll use this to impute missing categories and to fix orphaned merchant IDs.

**Most of the dictionary is provided below. Three entries are missing — fill them in.**
Look at the merchants table and the categories table to figure out the correct category_id for each.

In [106]:
# Review the merchants and categories tables before filling in the dictionary
print('Merchants table:')
print(merchants.to_string())
print('\nCategories table:')
print(categories.to_string())

Merchants table:
   merchant_id        merchant_name            industry           city  is_online
0    M_PAYROLL              Payroll             Banking         Online          1
1       M_SIDE          Side Hustle             Banking         Online          1
2    M_BANKINT        Bank Interest             Banking         Online          1
3     M_CC_PAY         CC Processor             Banking         Online          1
4       M_401K      401(k) Provider             Banking         Online          1
5        M0001            FreshMart           Groceries     Neenah, WI          0
6        M0002            FreshMart           Groceries  Green Bay, WI          0
7        M0003            QuickStop           Groceries    Oshkosh, WI          0
8        M0004            QuickStop           Groceries   Kimberly, WI          0
9        M0005            BrightGas  Gas/Transportation   Appleton, WI          0
10       M0006            BrightGas  Gas/Transportation    Menasha, WI          0

In [107]:
# Merchant-to-category lookup dictionary
# Fill in the three entries marked with '????' below.

merchant_to_category = {
    # Groceries
    'M0001': 'GROC',  # FreshMart - Neenah
    'M0002': 'GROC',  # FreshMart - Green Bay
    'M0003': 'GROC',  # QuickStop - Oshkosh
    'M0004': 'GROC',  # QuickStop - Kimberly
    # Gas / Transportation
    'M0005': 'GAS',   # BrightGas - Appleton
    'M0006': 'GAS',   # BrightGas - Menasha
    'M0007': 'GAS',   # BrightGas - Kimberly
    'M0008': 'TRANS', # MetroTransit - Appleton  <-- FILL THIS IN
    # Dining
    'M0009': 'DINE',  # BrewHouse - Neenah
    'M0010': 'DINE',  # BrewHouse - Appleton
    # Subscriptions
    'M0011': 'SUBS',  # StreamFlix
    # Phone / Internet
    'M0012': 'PHONE',  # GigaNet  <-- FILL THIS IN
    # Utilities
    'M0013': 'UTIL',  # CityPower
    # Rent
    'M0014': 'RENT',  # FoxRiver Apartments
    # Insurance
    'M0015': 'INS',   # HealthPlus
    # Health
    'M0016': 'HLTH',  # PharmaCare - Oshkosh
    'M0017': 'HLTH',  # PharmaCare - Appleton
    # Auto
    'M0018': 'AUTO',  # AutoWorks
    # Shopping
    'M0019': 'SHOP',  # StyleStreet
    # Education
    'M0020': 'EDU',   # BookNook  <-- FILL THIS IN
    # Entertainment
    'M0021': 'ENT',   # CoffeeCo - Neenah
    'M0022': 'ENT',   # CoffeeCo - Kimberly
    # System merchants
    'M_PAYROLL': 'PAY',
    'M_SIDE':    'SIDE',
    'M_BANKINT': 'INT',
    'M_CC_PAY':  'CC_PAY',
    'M_401K':    'RET401',
}

# Verify all entries are filled in before continuing
missing_entries = [k for k, v in merchant_to_category.items() if v == '????']
if missing_entries:
    print(f'Still missing: {missing_entries} — fill these in before moving to Part 5.')
else:
    print(f'Dictionary complete: {len(merchant_to_category)} entries')

Dictionary complete: 27 entries


---
## Part 5: Fix the Problems

Apply all fixes to a **copy** of the transactions DataFrame — never modify the original.
Write one fix per cell and add a comment explaining what each cell does.

Fix every problem you identified in Part 3. The order below is a reasonable sequence,
but adjust as needed based on what you found.

**Reminders:**
- Use `.loc[]` with a boolean mask for targeted replacements
- Use `.map()` to apply a dictionary lookup across a column
- Use `.fillna()` to fill missing values
- Calculate medians from **valid rows only** — exclude the rows you're about to fix

In [108]:
# Always work on a copy — never modify the original
# Your code here
transactions_clean = transactions.copy()
print('Working on a copy. Shape:', transactions_clean.shape)

Working on a copy. Shape: (1378, 9)


In [109]:
# Fix 1: Impute missing/empty category_id values
# Your code here
transactions_clean.loc[transactions_clean['category_id'].isnull() | (transactions_clean['category_id'] == ''), 'category_id'] = 'UNKNOWN'

In [110]:
# Fix 2: Address implausible amount values
# Your code here
print("DINE")
print(transactions_clean.loc[(transactions_clean['category_id'] == 'DINE') & (transactions_clean['amount'] < -91), 'amount'])
print("ENT")
print(transactions_clean.loc[(transactions_clean['category_id'] == 'ENT') & (transactions_clean['amount'] < -70), 'amount'])
print("GROC")
print(transactions_clean.loc[(transactions_clean['category_id'] == 'GROC') & (transactions_clean['amount'] < -122), 'amount'])

DINE
226   -143.20
499   -210.00
557   -107.64
814   -162.75
935   -178.30
Name: amount, dtype: float64
ENT
99     -187.50
800     -95.40
876     -76.16
1165   -220.00
1237    -79.84
1271   -145.90
Name: amount, dtype: float64
GROC
48    -124.50
121   -842.50
483   -132.82
616   -131.77
969   -132.73
977   -137.72
Name: amount, dtype: float64


In [111]:
# Fix 3: Impute missing amounts
# Your code here

transactions_clean.loc[transactions_clean['amount'].isnull(), 'amount'] = transactions_clean['amount'].median()

In [112]:
# Fix 4: Correct orphaned merchant IDs
# Hint: build a reverse lookup — category -> sorted list of valid merchant IDs,
# then assign the first valid merchant for that category
# Your code here

valid_ids = set(merchants['merchant_id'])

# keep only valid merchants per category
valid_map = transactions_clean[transactions_clean['merchant_id'].isin(valid_ids)] \
    .groupby('category_id')['merchant_id'] \
    .apply(list)

# pick first valid merchant per category
first_valid = valid_map.str[0]

# find orphaned merchant IDs
orphan_mask = ~transactions_clean['merchant_id'].isin(valid_ids)

# replace using category lookup
transactions_clean.loc[orphan_mask, 'merchant_id'] = transactions_clean.loc[orphan_mask, 'category_id'].map(first_valid)

---
## Part 6: Verify

Re-run your checks from Part 3 on the cleaned DataFrame.
Every problem you identified and fixed should now return 0.

Write a check for each problem type you fixed and print a clear pass/fail message for each one.

In [113]:
# Verify all fixes — one check per problem type
# Your code here
print("Missing values by column: category_id", (transactions_clean['category_id'] == '').sum())
print("Missing values by column:", transactions_clean.isnull().sum())


print('=== VERIFICATION ===')

# Check 1: No missing amounts
missing = transactions_clean['amount'].isnull().sum()
print(f'Missing amounts: {missing}  (expected 0)')

# Check 3: No orphaned merchant_ids (excluding UNKNOWN which we set intentionally)
known_valid = valid_ids | {'UNKNOWN'}
still_orphaned = (~transactions_clean['merchant_id'].isin(known_valid)).sum()
print(f'Orphaned merchant_ids remaining: {still_orphaned}  (expected 0)')


Missing values by column: category_id 0
Missing values by column: transaction_id      0
date                0
account_id          0
transaction_type    0
category_id         0
merchant_id         0
amount              0
payment_method      0
description         0
dtype: int64
=== VERIFICATION ===
Missing amounts: 0  (expected 0)
Orphaned merchant_ids remaining: 0  (expected 0)


In [116]:
# Save the cleaned transactions file
# Your code here
from pathlib import Path

output_path = Path('transactions_clean.csv').resolve()
transactions_clean.to_csv(output_path, index=False)

---
## Part 7: Reflection

Answer the following questions in this markdown cell.

1. **How many total problems did you find?** List each problem type, which transactions were affected, and how you identified them.

2. **For the implausible amount values**: what thresholds did you choose and why? What information from the audit led you to those numbers?

3. **For the missing category_id values**: why is using the merchant_to_category dictionary a more reliable fix than simply dropping those rows?

4. **For the orphaned merchant IDs**: you assigned a valid merchant based on the transaction's category. What is one limitation of this approach?

5. **Looking ahead to the project**: what would go wrong in a running balance calculation if you used the uncleaned data instead of your cleaned version?

*The problem types were completeness, accuracy, and validity. There were three orphaned merchant IDS, 2 missing amounts, and questionable amounts for three categories. I identified the merchant IDs by comparing the transaction IDs to the Merchant tables' IDs. Amount nulls were found using is null. The questionable amounts were found while investigating the min, median, max, and count for each category.

*I based the implausible amounts on the distance from the min to the median, which led me to -91, -70, and -122. I then looked through the transactions for each and found that most of the amounts are within the standards.

Using the merchant_to_category dictionary is a more reliable fix because it preserves the data. 

Assigning a valid merchant based on the transaction category has the limitation of possibly forcing the transaction and misrepresenting it.

Things like the median would be messed up and not represent the data correctly if you use unclean data.

